# Part 7 · Notebook 02 — Directional momentum

**Sessions:** S2 (Directional momentum group) · [Lesson plan](../../docs/lessons/PART_07_STRATEGY_LIBRARY.md) · graded labs in [`labs/part07/`](../../labs/part07/)

**You will:**
1. Write time-series momentum with volatility targeting.
2. Write the exits of a Donchian breakout (a small state machine).
3. Build 12-1 cross-sectional momentum.
4. See where momentum works, and where it bleeds, using the known regimes.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic, built from regimes you know, and every strategy here is a **hypothesis** with a first-look evaluation: the honest backtest comes in Part 8.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p7lib.py is in notebooks/part07/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p7lib as p

p.use_course_style()

In [ ]:
bars = p.regime_market()
o, h, l, c = (bars[k].to_numpy() for k in ("open", "high", "low", "close"))

## 1. Time-series momentum (TSMOM)

Be long if the asset is up over the last year, short if down, and size the position so its risk is constant: `sign(C[t]/C[t−lookback] − 1) × target_vol / realized_vol`, clipped to `±max_lev`. Realized vol is the rolling standard deviation of daily log returns over `vol_n` days, annualized. Without the vol scaling, momentum's drawdowns come in the volatile periods.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def tsmom(close, lookback=252, vol_n=60, target_vol=0.10, max_lev=2.0):
    past = np.full(close.shape, np.nan)
    past[lookback:] = close[lookback:] / close[:-lookback] - 1
    r = np.full(close.shape, np.nan)
    r[1:] = np.diff(np.log(close))
    vol = pd.Series(r).rolling(vol_n).std().to_numpy() * np.sqrt(252)
    return ...                                    # ✍️ direction × target / vol, clipped to ±max_lev

mine = p.attempt(tsmom, c)
mine = p.check("tsmom", mine, p.tsmom(c))
p.summary(p.quick_eval(mine, o))

## 2. The Donchian breakout

The Turtle rule, long and short. Channels use the **prior** bars only (today's bar can't be part of the level it breaks): entries on a break of the prior `entry_n`-bar high or low, exits on a break of the prior `exit_n`-bar low (for a long) or high (for a short). The flat branch is written; write the two exits.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def donchian(high, low, close, entry_n=20, exit_n=10):
    pos = np.zeros(close.size)
    for t in range(max(entry_n, exit_n), close.size):
        prev = pos[t - 1]
        if prev == 0:
            if close[t] > high[t - entry_n:t].max():
                pos[t] = 1
            elif close[t] < low[t - entry_n:t].min():
                pos[t] = -1
        elif prev == 1:
            pos[t] = ...                          # ✍️ 0 if close < the prior exit_n lows' minimum, else stay long
        else:
            pos[t] = ...                          # ✍️ 0 if close > the prior exit_n highs' maximum, else stay short
    return pos

mine = p.attempt(donchian, h, l, c)
mine = p.check("donchian", mine, p.donchian_breakout(h, l, c))
print(f"long {np.mean(np.asarray(mine) == 1):.0%}, short {np.mean(np.asarray(mine) == -1):.0%}, flat {np.mean(np.asarray(mine) == 0):.0%} of bars")

## 3. Where momentum works

Sharpe ratio of each strategy's first-look P&L **inside** each of the four known regimes. This table, not the overall Sharpe, is what the strategy spec's "works when / fails when" section is made of.

In [ ]:
signals = {"TSMOM": p.tsmom(c), "Donchian 20/10": p.donchian_breakout(h, l, c), "buy & hold": np.ones(len(c))}
table = p.regime_table(bars, signals)
display(table.round(2))
eq = {k: np.cumprod(1 + p.quick_eval(v, o)["pnl"]) for k, v in signals.items()}
fig, ax = plt.subplots(figsize=(11, 3.8))
for k, v in eq.items():
    ax.plot(bars.index, v, label=k)
ax.set_yscale("log"); ax.set_title("First-look equity (not a backtest)"); ax.legend(); plt.show()

## 4. Cross-sectional momentum (12-1)

Across a universe, buy the recent **winners**. Momentum is measured from 12 months ago to 1 month ago, skipping the latest month (which tends to reverse): `mom = closes.shift(1) / closes.shift(12) − 1` on month-end prices. Rank across assets each month (`rank(axis=1, pct=True)`) and equal-weight the names ranked strictly above `1 − top_q`.

In [ ]:
uni = p.momentum_universe()
uni.iloc[:, :6].plot(figsize=(10, 3.5), legend=False, logy=True, title="12 assets with persistent (but slowly changing) drifts"); plt.show()

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def xs_momentum(closes, top_q=0.25):
    mom = ...                                     # ✍️ 12-1 momentum
    rank = mom.rank(axis=1, pct=True)
    w = (rank > 1 - top_q + 1e-12).astype(float)
    return w.div(w.sum(axis=1), axis=0).fillna(0.0)

mine = p.attempt(xs_momentum, uni)
mine = p.check("cross-sectional momentum", mine, p.cross_sectional_momentum(uni))
fwd = uni.pct_change().shift(-1)                  # next month's return: weights set at month end t earn month t+1
top, avg = (mine * fwd).sum(axis=1)[13:-1], fwd.mean(axis=1)[13:-1]
sr = lambda x: x.mean() / x.std() * np.sqrt(12)
print(f"winners: Sharpe {sr(top):.2f};  equal weight: {sr(avg):.2f};  winners minus average: {sr(top - avg):.2f}")

In this universe expected returns are persistent by construction, which is exactly the assumption momentum bets on. In real markets the premium is smaller, decays after publication, and crashes when the market rebounds sharply (Daniel & Moskowitz, 2016).

## Wrap-up

* Momentum earns in trends and bleeds in ranges: say so in the spec, and size by volatility.
* Breakout channels use prior bars only; momentum signals skip the most recent month.
* Graded version: `labs/part07/week23_framework_momentum` (TSMOM, Donchian vectorized and live-style, 12-1, dual momentum).